In [1]:
# ============================================================
# Check final fairness dataset inventory and folder structure
# ============================================================

import os
from pathlib import Path

import pandas as pd

BASE = Path(os.environ.get("A100", "/home/tahiti/DataGenaration")).expanduser().resolve()
DATA_DIR = BASE / "fairness_datasets"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("=" * 80)
print("BASE:", BASE)
print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("=" * 80)

# ------------------------------------------------------------
# 1. Folder tree
# ------------------------------------------------------------

def human_size(n):
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"

def folder_size(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

print("\nFOLDER STRUCTURE:")
for root in [RAW_DIR, PROCESSED_DIR]:
    print(f"\n{root} | size={human_size(folder_size(root))}")
    if root.exists():
        for p in sorted(root.iterdir()):
            if p.is_dir():
                print(f"  [DIR]  {p.name:<35} size={human_size(folder_size(p))}")
            else:
                print(f"  [FILE] {p.name:<35} size={human_size(p.stat().st_size)}")
    else:
        print("  MISSING")

# ------------------------------------------------------------
# 2. Processed CSV inventory
# ------------------------------------------------------------

expected_sensitive = {
    "adult_processed.csv": ["sex", "race", "age", "age_group", "education", "marital_status"],
    "compas_processed.csv": ["sex", "race", "age", "age_cat", "age_group"],
    "german_credit_processed.csv": ["personal_status_sex", "age", "age_group", "foreign_worker"],
    "bank_marketing_processed.csv": ["age", "age_group", "marital", "education"],
    "communities_crime_processed.csv": ["racepctblack", "racePctWhite", "racePctHisp", "racePctAsian", "black_group", "white_group", "hisp_group", "asian_group"],
    "default_credit_card_processed.csv": ["SEX", "EDUCATION", "MARRIAGE", "AGE", "age_group"],
    "student_performance_processed.csv": ["sex", "age", "age_group", "address", "famsize", "Pstatus"],
    "heart_disease_processed.csv": ["sex", "age", "age_group"],
    "law_school_processed.csv": ["race", "sex", "gender", "target"],
}

rows = []

print("\n" + "=" * 80)
print("PROCESSED DATASETS")
print("=" * 80)

csv_files = sorted(PROCESSED_DIR.glob("*.csv")) if PROCESSED_DIR.exists() else []

if not csv_files:
    print("No processed CSV files found.")
else:
    for csv_path in csv_files:
        print("\n" + "-" * 80)
        print("FILE:", csv_path.name)
        print("PATH:", csv_path)
        print("SIZE:", human_size(csv_path.stat().st_size))

        try:
            df_head = pd.read_csv(csv_path, nrows=5)
            df_full = pd.read_csv(csv_path)

            n_rows, n_cols = df_full.shape
            cols = list(df_full.columns)

            has_target = "target" in cols
            target_info = "NO TARGET"

            if has_target:
                vc = df_full["target"].value_counts(dropna=False).head(10)
                target_info = "; ".join([f"{k}: {v}" for k, v in vc.items()])

            known_sensitive = expected_sensitive.get(csv_path.name, [])
            found_sensitive = [c for c in known_sensitive if c in cols]

            possible_sensitive = []
            for c in cols:
                cl = c.lower()
                if any(key in cl for key in [
                    "sex", "gender", "race", "age", "marital", "education",
                    "foreign", "group", "black", "white", "hisp", "asian",
                    "address", "famsize", "pstatus", "marriage"
                ]):
                    possible_sensitive.append(c)

            possible_sensitive = list(dict.fromkeys(possible_sensitive))

            print("SHAPE:", df_full.shape)
            print("HAS target:", has_target)
            print("TARGET distribution:", target_info)
            print("FOUND expected sensitive:", found_sensitive)
            print("POSSIBLE sensitive columns:", possible_sensitive[:20])
            print("COLUMNS:")
            print(cols)
            print("HEAD:")
            display(df_head)

            rows.append({
                "file": csv_path.name,
                "path": str(csv_path),
                "size": human_size(csv_path.stat().st_size),
                "rows": n_rows,
                "cols": n_cols,
                "has_target": has_target,
                "target_distribution_top": target_info,
                "found_expected_sensitive": ", ".join(found_sensitive),
                "possible_sensitive_columns": ", ".join(possible_sensitive[:30]),
                "all_columns": ", ".join(cols),
            })

        except Exception as e:
            print("[ERROR reading]", csv_path.name, e)

            rows.append({
                "file": csv_path.name,
                "path": str(csv_path),
                "size": human_size(csv_path.stat().st_size),
                "rows": None,
                "cols": None,
                "has_target": False,
                "target_distribution_top": f"ERROR: {e}",
                "found_expected_sensitive": "",
                "possible_sensitive_columns": "",
                "all_columns": "",
            })

# ------------------------------------------------------------
# 3. Save inventory
# ------------------------------------------------------------

inventory = pd.DataFrame(rows)

inventory_path = DATA_DIR / "dataset_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)

if len(inventory) > 0:
    display(inventory[[
        "file",
        "rows",
        "cols",
        "has_target",
        "found_expected_sensitive",
        "possible_sensitive_columns",
    ]])
else:
    print("Inventory is empty.")

print("\nSaved inventory to:")
print(inventory_path)

# ------------------------------------------------------------
# 4. Recommended final experimental set
# ------------------------------------------------------------

recommended = [
    "adult_processed.csv",
    "compas_processed.csv",
    "german_credit_processed.csv",
    "bank_marketing_processed.csv",
    "communities_crime_processed.csv",
    "default_credit_card_processed.csv",
    "student_performance_processed.csv",
    "heart_disease_processed.csv",
    "law_school_processed.csv",
]

existing = [p.name for p in csv_files]
available_recommended = [x for x in recommended if x in existing]
missing_recommended = [x for x in recommended if x not in existing]

print("\n" + "=" * 80)
print("RECOMMENDED EXPERIMENTAL SET")
print("=" * 80)
print("Available:")
for x in available_recommended:
    print("  +", x)

print("\nMissing:")
for x in missing_recommended:
    print("  -", x)

print("\nDONE")

BASE: /home/tahiti/DataGenaration
DATA_DIR: /home/tahiti/DataGenaration/fairness_datasets
RAW_DIR: /home/tahiti/DataGenaration/fairness_datasets/raw
PROCESSED_DIR: /home/tahiti/DataGenaration/fairness_datasets/processed

FOLDER STRUCTURE:

/home/tahiti/DataGenaration/fairness_datasets/raw | size=20.16 MB
  [DIR]  acs_folktables                      size=4.72 KB
  [DIR]  adult                               size=5.71 MB
  [DIR]  bank_marketing                      size=5.39 MB
  [DIR]  communities_crime                   size=1.08 MB
  [DIR]  compas                              size=2.43 MB
  [DIR]  default_credit_card                 size=5.28 MB
  [DIR]  dutch_census                        size=0.00 B
  [DIR]  german_credit                       size=82.49 KB
  [DIR]  heart_disease                       size=18.03 KB
  [DIR]  law_school                          size=0.00 B
  [DIR]  student_performance                 size=170.08 KB

/home/tahiti/DataGenaration/fairness_datasets/process

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,target,age_group
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0,2
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0,3
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0,2
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0,3
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0,0



--------------------------------------------------------------------------------
FILE: bank_marketing_processed.csv
PATH: /home/tahiti/DataGenaration/fairness_datasets/processed/bank_marketing_processed.csv
SIZE: 3.57 MB
SHAPE: (45211, 18)
HAS target: True
TARGET distribution: 0: 39922; 1: 5289
FOUND expected sensitive: ['age', 'age_group', 'marital', 'education']
POSSIBLE sensitive columns: ['age', 'marital', 'education', 'age_group']
COLUMNS:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'target', 'age_group']
HEAD:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,target,age_group
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0,3
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0,2
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0,2
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0,0



--------------------------------------------------------------------------------
FILE: communities_crime_processed.csv
PATH: /home/tahiti/DataGenaration/fairness_datasets/processed/communities_crime_processed.csv
SIZE: 1.02 MB
SHAPE: (1994, 128)
HAS target: True
TARGET distribution: 0: 1001; 1: 993
FOUND expected sensitive: ['racepctblack', 'racePctWhite', 'racePctHisp', 'racePctAsian', 'black_group', 'white_group', 'hisp_group', 'asian_group']
POSSIBLE sensitive columns: ['racepctblack', 'racePctWhite', 'racePctAsian', 'racePctHisp', 'agePct12t21', 'agePct12t29', 'agePct16t24', 'agePct65up', 'pctWWage', 'whitePerCap', 'blackPerCap', 'AsianPerCap', 'HispPerCap', 'PctForeignBorn', 'PctPolicWhite', 'PctPolicBlack', 'PctPolicHisp', 'PctPolicAsian', 'black_group', 'white_group']
COLUMNS:
['population', 'householdsize', 'racepctblack', 'racePctWhite', 'racePctAsian', 'racePctHisp', 'agePct12t21', 'agePct12t29', 'agePct16t24', 'agePct65up', 'numbUrban', 'pctUrban', 'medIncome', 'pctWWage', 

...


--------------------------------------------------------------------------------
FILE: student_performance_processed.csv
PATH: /home/tahiti/DataGenaration/fairness_datasets/processed/student_performance_processed.csv
SIZE: 115.91 KB
SHAPE: (1044, 36)
HAS target: True
TARGET distribution: 1: 814; 0: 230
FOUND expected sensitive: ['sex', 'age', 'age_group', 'address', 'famsize', 'Pstatus']
POSSIBLE sensitive columns: ['sex', 'age', 'address', 'famsize', 'Pstatus', 'age_group']
COLUMNS:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'subject', 'target', 'age_group']
HEAD:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,Dalc,Walc,health,absences,G1,G2,G3,subject,target,age_group
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,1,1,3,6,5,6,6,mat,0,2
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,1,1,3,4,5,5,6,mat,0,1
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,2,3,3,10,7,8,10,mat,1,0
3,GP,F,15,U,GT3,T,4,2,health,services,...,1,1,5,2,15,14,15,mat,1,0
4,GP,F,16,U,GT3,T,3,3,other,other,...,1,2,5,4,6,10,10,mat,1,0



SUMMARY TABLE


,file,rows,cols,has_target,found_expected_sensitive,possible_sensitive_columns
0,adult_processed.csv,48842,16,True,"sex, race, age, age_group, education, marital_...","age, education, education_num, marital_status,..."
1,bank_marketing_processed.csv,45211,18,True,"age, age_group, marital, education","age, marital, education, age_group"
2,communities_crime_processed.csv,1994,128,True,"racepctblack, racePctWhite, racePctHisp, raceP...","racepctblack, racePctWhite, racePctAsian, race..."
3,compas_processed.csv,7214,11,True,"sex, race, age, age_cat, age_group","age, age_cat, sex, race, age_group"
4,default_credit_card_processed.csv,30000,25,True,"SEX, EDUCATION, MARRIAGE, AGE, age_group","SEX, EDUCATION, MARRIAGE, AGE, age_group"
5,german_credit_processed.csv,1000,22,True,"personal_status_sex, age, age_group, foreign_w...","personal_status_sex, age, foreign_worker, age_..."
6,heart_disease_processed.csv,303,16,True,"sex, age, age_group","age, sex, age_group"
7,student_performance_processed.csv,1044,36,True,"sex, age, age_group, address, famsize, Pstatus","sex, age, address, famsize, Pstatus, age_group"



Saved inventory to:
/home/tahiti/DataGenaration/fairness_datasets/dataset_inventory.csv

RECOMMENDED EXPERIMENTAL SET
Available:
  + adult_processed.csv
  + compas_processed.csv
  + german_credit_processed.csv
  + bank_marketing_processed.csv
  + communities_crime_processed.csv
  + default_credit_card_processed.csv
  + student_performance_processed.csv
  + heart_disease_processed.csv

Missing:
  - law_school_processed.csv

DONE
